In [2]:
import os
from autogen.agentchat import UserProxyAgent, AssistantAgent, GroupChat, GroupChatManager
from autogen.coding import LocalCommandLineCodeExecutor
from dotenv import load_dotenv
from openai import AzureOpenAI
import json
import pandas as pd
import numpy as np
load_dotenv()

azure_gpt4o = {
    "api_type": "azure",
    "model": os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'),
    "api_key": os.getenv('OPENAI_API_KEY'),
    "base_url": os.getenv('AZURE_OPENAI_ENDPOINT'),
    "api_version": os.getenv('OPENAI_API_VERSION')
}

flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.


In [3]:
#print(os.getenv('OPENAI_API_VERSION'))
blandai_data_dir = os.path.join(os.getcwd(), 'blandai-data')
codebook_file = os.path.join(os.getcwd(), 'transcripts/codebook.json')
transcripts_file = os.path.join(blandai_data_dir, 'blandai_transcripts_2025-01-06.json')
#sample_output_file = os.path.join(data_dir, 'example_output.csv')
#print(sample_output_file)
print(codebook_file)
print(blandai_data_dir)
print(transcripts_file)

/Users/kaiyrbekovk2/Repositories/ai-agent-based-survey/transcripts/codebook.json
/Users/kaiyrbekovk2/Repositories/ai-agent-based-survey/blandai-data
/Users/kaiyrbekovk2/Repositories/ai-agent-based-survey/blandai-data/blandai_transcripts_2025-01-06.json


In [11]:
def get_QA_and_codebook(codebook_file):
    qa_list = []
    with open(codebook_file, 'r') as file:
        codebook = json.load(file)
        question_count = 1
        for code, content in codebook.items():  
            question = content['question']
            if code == 'AGE7':
                question = 'How old are you?'
            qa_list.append(f"Question {question_count}: {question}\nResponse Options: ")
            ans_list = []
            for ans, ans_id in content['clean_response_text_to_id'].items():
                ans_list.append(f"{ans}")
            if code in ['HH01S', 'HH25S', 'HH612S', 'HH1317S', 'HH18OVS', 'PHYS11_TEMP']:
                ans_list = ['Numeric Value']
                
            qa_list.append('; '.join(ans_list))
            qa_list.append("\n")
            question_count += 1
    return ''.join(qa_list), codebook
QA_details, codebook = get_QA_and_codebook(codebook_file)
print(QA_details)

Question 1: How old are you?
Response Options: 18-24; 25-34; 35-44; 45-54; 55-64; 65-74; 75+; Under 18
Question 2: Are you male or female
Response Options: Unknown; Male; Female; Not sure; REFUSED
Question 3: Race/ethnicity
Response Options: White, non-Hispanic; Black, non-Hispanic; Hispanic; Other, non-Hispanic; DON'T KNOW; Removed for disclosure risk; REFUSED
Question 4: Household income
Response Options: Under $10,000; $10,000 to under $20,000; $20,000 to under $30,000; $30,000 to under $40,000; $40,000 to under $50,000; $50,000 to under $75,000; $75,000 to under $100,000; $100,000 to under $150,000; $150,000 or more; DON'T KNOW; REFUSED
Question 5: What is the highest level of school you have completed?
Response Options: No HS diploma; HIGH SCHOOL GRADUATE - high school DIPLOMA or the equiva; Some college, no degree; Associate degree; Bachelors degree; Masters degree; Professional or Doctorate degree; Not sure; REFUSED
Question 6: Household size (including children)
Response Option

In [14]:
client = AzureOpenAI(
  api_key = os.getenv('OPENAI_API_KEY'),  
  api_version = os.getenv('OPENAI_API_VERSION'),
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
)



role_description = f''' You are a helpful assistant that reads conversation transcipt and deduces responses given by user to each question.
                        You return an output with the responses for each question in order of they appear in the question list.
                        The returned output needs to be a single line, with individual responses separated by semicolons and no other punctuation. 
                        Each deduced response for a question should be strictly selected from corresponding Reponse Options. 
                        If Reponse Options includes Numeric Value as an option, deduce actual numeric value from conversation.
                        Each question should have an answer, if question has Numeric Value as an option you couldn't deduce the response put 'NaN'.
                        
                        Here is the list of questions with respective comma separated response options:
                        {QA_details}
                        '''
#print(role_description)



with open(transcripts_file, 'r') as file:
    transcripts = json.load(file)

task_prompt = f"""Help me to understand the following conversation transcript : 

{transcripts["0"]}"""
#print(task_prompt)

conversation=[{"role": "system", "content": role_description}]
userid_to_answers = {}

for user_id, transcript in transcripts.items():
    task_prompt = f"""Help me to understand the following conversation transcript : 
                    {transcripts[user_id].replace('user:', 'surveyee:')}"""
    conversation.append({"role": "user", "content": task_prompt})

    #print(task_prompt)
    response = client.chat.completions.create(
        model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
        messages=conversation
    )
    userid_to_answers[user_id] = response.choices[0].message.content
    print(userid_to_answers[user_id])
    conversation.pop()

35-44; Female; White, non-Hispanic; $20,000 to under $30,000; Some college, no degree; Four persons; 0; 1; 0; 2; 1; Once a month; Once a month; Not at all or less than 1 day; Not at all or less than 1 day; Not at all or less than 1 day; 1-2 days; 1-2 days; No, I did not work for pay last week; Good; Yes; No; Yes; No; Yes; Yes; No; No; Yes; Yes; Yes; Yes; 98.6
65-74; Male; White, non-Hispanic; $150,000 or more; Bachelors degree; Two persons; 0; 0; 0; 0; 2; A few times a month; A few times a month; Not at all or less than 1 day; Not at all or less than 1 day; Not at all or less than 1 day; Not at all or less than 1 day; Not at all or less than 1 day; No, I did not work for pay last week; Very good; No; No; No; No; No; No; No; No; No; No; No; Yes; 98.5
 65-74; Male; Other, non-Hispanic; $150,000 or more; Bachelors degree; Two persons; 0; 0; 0; 0; 2; A few times a month; A few times a month; Not at all or less than 1 day; Not at all or less than 1 day; Not at all or less than 1 day; 1-2 da

In [5]:
answers_as_list = []
for user_id in sorted(list(userid_to_answers.keys())):
    user_answers = [part.strip() for part in userid_to_answers[user_id].split(';')]
    answers_as_list.append(user_answers)

In [6]:
question_code_to_answers = {}
i = 0
#print(codebook['SOC1']['answer_to_answer_id']['Some'])
for code, val in codebook.items():
    print(code)
    question_code_to_answers[code] = []
    for user_answers in answers_as_list:
        ans_text = user_answers[i]
        question_code_to_answers[code].append(codebook[code]['clean_response_text_to_id'][ans_text])
    i += 1

AGE7
GENDER
RACETH
HHINCOME
EDUCATION
HHSIZE1
HH01S


KeyError: 'NaN'

In [ ]:
df = pd.DataFrame(question_code_to_answers)

In [ ]:
df.to_csv('deduced.csv')